# ResNet-18 직접 구현하기 (ImageNet)

In [6]:
import torch
import torch.nn as nn



논문 Figure 2 참고


In [7]:
class BasicBlock(nn.Module):
    def __init__(self, in_channels, out_channels, stride=1) :
        super().__init__()
        self.block = nn.Sequential(
            nn.Conv2d(in_channels=in_channels,
                      out_channels=out_channels,
                      kernel_size=3,
                      stride=stride,
                      padding=1,
                      bias=False),
            nn.BatchNorm2d(out_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(in_channels=out_channels,
                      out_channels=out_channels,
                      kernel_size=3,
                      stride=1,
                      padding=1,
                      bias=False),
            nn.BatchNorm2d(out_channels),
            )  
        
        self.relu = nn.ReLU(inplace=True)

        if stride != 1 or in_channels != out_channels : 
            self.identity = nn.Sequential(
                nn.Conv2d(in_channels=in_channels,
                          out_channels=out_channels,
                          kernel_size=1,
                          stride=stride,
                          bias=False
                          ),
                nn.BatchNorm2d(out_channels)
            )
        else :
            self.identity = nn.Identity()
    

    def forward(self, x) :
        out = self.block(x)
        x = self.identity(x)
        out += x

        out = self.relu(out)
        return out
    

class BottleNeckBlock(nn.Module) :
    def __init__(self, in_channels, out_channels, stride=1) :
        super().__init__()
        mid_channels = out_channels // 4
        self.block = nn.Sequential(
            nn.Conv2d(in_channels=in_channels,
                      out_channels=mid_channels,
                      kernel_size=1,
                      stride=1,
                      bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(in_channels=mid_channels,
                      out_channels=mid_channels,
                      kernel_size=3,
                      stride=stride,
                      padding=1,
                      bias=False),
            nn.BatchNorm2d(mid_channels),
            nn.ReLU(inplace=True),

            nn.Conv2d(in_channels=mid_channels,
                      out_channels=out_channels,
                      kernel_size=1,
                      stride=1,
                      bias=False),
            nn.BatchNorm2d(out_channels)
        )

        self.relu = nn.ReLU(inplace=True)

        if stride != 1 or in_channels != out_channels : 
            self.identity = nn.Sequential(
                nn.Conv2d(in_channels=in_channels,
                          out_channels=out_channels,
                          kernel_size=1,
                          stride=stride,
                          bias=False
                          ),
                nn.BatchNorm2d(out_channels)
            )
        else :
            self.identity = nn.Identity()

    def forward(self, x) :
        out = self.block(x)
        x = self.identity(x)
        out += x

        out = self.relu(out)
        return out

## ResNet-18 구현

논문 Table 1 참고

In [8]:
class ResNet18(nn.Module):
    def __init__(self, num_classes=1000):
        super().__init__()
        self.conv1 = nn.Conv2d(in_channels=3,
                               out_channels=64,
                               kernel_size=7,
                               stride=2,
                               padding=3,
                               bias=False)
        self.bn1 = nn.BatchNorm2d(64)
        self.relu = nn.ReLU(inplace=True)
        self.maxpool = nn.MaxPool2d(kernel_size=3, 
                                    stride=2, 
                                    padding=1
        )

        self.conv2_x = nn.Sequential(
            BasicBlock(64, 64),
            BasicBlock(64, 64)
        )

        self.conv3_x = nn.Sequential(
            BasicBlock(64, 128, stride=2),
            BasicBlock(128, 128)
        )

        self.conv4_x = nn.Sequential(
            BasicBlock(128, 256, stride=2),
            BasicBlock(256, 256)
        )

        self.conv5_x = nn.Sequential(
            BasicBlock(256, 512, stride=2),
            BasicBlock(512, 512)
        )

        self.avgpool = nn.AdaptiveAvgPool2d((1, 1)) 
        self.fc = nn.Linear(1 * 1 * 512, num_classes)

    def forward(self, x):
        x = self.maxpool(self.relu(self.bn1(self.conv1(x))))
        x = self.conv2_x(x)
        x = self.conv3_x(x)
        x = self.conv4_x(x)
        x = self.conv5_x(x)
        x = self.avgpool(x)         # [B, 512, 1, 1]
        x = torch.flatten(x, 1)     # [B, 512]
        x = self.fc(x)
        return x

In [9]:
model = ResNet18(num_classes=1000)
r = torch.randn(1, 3, 224, 224)
out = model(r)
print(out.shape) 

torch.Size([1, 1000])
